# Supplemental Feature Plots

Kendra Wyant  
July 16, 2024

### Set Up Environment

In [ ]:
#| message: false
#| warning: false

# handle conflicts
options(conflicts.policy = "depends.ok")
devtools::source_url("https://github.com/jjcurtin/lab_support/blob/main/fun_ml.R?raw=true")

ℹ SHA-1 hash of file is "77e91675366f10788c6bcb59fa1cfc9ee0c75281"

In [ ]:
#| message: false
#| warning: false

suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(tidymodels))
library(kableExtra, exclude = "group_rows")


theme_set(theme_classic()) 

In [ ]:
#| output: false

devtools::source_url("https://github.com/jjcurtin/lab_support/blob/main/format_path.R?raw=true")

ℹ SHA-1 hash of file is "a58e57da996d1b70bb9a5b58241325d6fd78890f"

ℹ SHA-1 hash of file is "75cc6f7b855da59c240908bd936834b4da01285b"

In [ ]:
path_processed <- format_path(str_c("studydata/risk/data_processed/lag"))
path_models_lag <- format_path(str_c("studydata/risk/models/lag"))
path_chtc <- format_path(str_c("studydata/risk/chtc/lag/train_xgboost_1week_0lag_nested_1_x_10_3_x_10_v2_main/input"))

source(here::here(path_chtc, "training_controls.R"))

### Read in Data

Raw Shapley values - (testing with 0 lag models first)

In [ ]:
shap_0 <- read_rds(here::here(path_models_lag, 
                              "outer_shaps_1week_0_v1_nested_main.rds")) |> 
  glimpse()

Rows: 77,138,381
Columns: 5
Groups: id_obs [270,081]
$ id_obs     <int> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,…
$ variable   <fct> lapse.p12.diff_count, lapse.p168.raw_count, lapse.p48.raw_c…
$ value      <dbl> -0.1202097358, -0.0959646162, -0.0357964309, -0.0459769163,…
$ rfvalue    <dbl> 0.016984509, -0.405306733, -0.296854932, -0.335092209, -0.0…
$ mean_value <dbl> 0.137626631, 0.138602994, 0.053141532, 0.065154397, 0.06426…

Read in features

In [ ]:
feat_0 <- read_csv(here::here(path_processed, "features_0lag_v1.csv.xz"),
                   show_col_types = FALSE) |> 
  glimpse()

Rows: 270,081
Columns: 285
$ label_num                       <dbl> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,…
$ subid                           <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,…
$ dttm_label                      <dttm> 2017-03-03 06:00:00, 2017-03-03 07:00…
$ lapse                           <chr> "no", "no", "no", "no", "no", "no", "n…
$ label_day                       <chr> "Fri", "Fri", "Fri", "Fri", "Fri", "Fr…
$ label_hour                      <chr> "other", "other", "other", "other", "o…
$ p12.l0.rratecount.count.lapse   <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…
$ p12.l0.dratecount.count.lapse   <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…
$ p24.l0.rratecount.count.lapse   <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…
$ p24.l0.dratecount.count.lapse   <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…
$ p48.l0.rratecount.count.lapse   <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…
$ p48.l0.dratecount.count.lapse   <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…
$ p72.l0.rrat

### Match raw feature to shaps

In [ ]:
feat_0 <- format_data(feat_0)  
rec <- recipe(y ~ ., data = feat_0) |>
    step_rm(subid) |>  
    step_impute_median(all_numeric_predictors()) |> 
    step_impute_mode(all_nominal_predictors()) |> 
    step_zv(all_predictors()) |> 
    step_dummy(all_factor_predictors()) |> 
    prep()

feat_0 <- bake(rec, new_data = NULL)

In [ ]:
feat_0 <- feat_0 |> 
  rename(id_obs = label_num) |> 
  select(-y)

feat_0 <- feat_0 |> 
  pivot_longer(cols = !starts_with("id"), names_to = "variable", values_to = "feature_score")

Update names of features to match SHAP

In [ ]:
clean_feature_names <- function(feat_name){
  new_name <- gsub(".l0", "", feat_name)
  new_name <- gsub("rratecount.count", "raw_count", new_name)
  new_name <- gsub("dratecount.count", "diff_count", new_name)
  new_name <- gsub("drecent_response", "diff_recent", new_name)
  new_name <- gsub("rrecent_response", "raw_recent", new_name)
  new_name <- gsub("dmin_response", "diff_min", new_name)
  new_name <- gsub("rmin_response", "raw_min", new_name)
  new_name <- gsub("dmax_response", "diff_max", new_name)
  new_name <- gsub("rmax_response", "raw_max", new_name)
  new_name <- gsub("dmedian_response", "diff_median", new_name)
  new_name <- gsub("rmedian_response", "raw_median", new_name)
  new_name <- gsub("label_", "", new_name)
  new_name <- gsub("demo_", "", new_name)
  new_name <- gsub("High.school.or.less", "high.school", new_name)
  new_name <- gsub("Some.college", "some.college", new_name)
  new_name <- gsub("Mon", "mon", new_name)
  new_name <- gsub("Tue", "tue", new_name)
  new_name <- gsub("Wed", "wed", new_name)
  new_name <- gsub("Thu", "thu", new_name)
  new_name <- gsub("Fri", "fri", new_name)
  new_name <- gsub("Sat", "sat", new_name)
  new_name <- gsub("Sun", "sun", new_name)
  new_name <- gsub("Never.Married", "never.married", new_name)
  new_name <- gsub("Never.Other", "never.other", new_name)
  new_name <- gsub("White.Caucasian", "caucasian", new_name)
  new_name <- gsub("Male", "male", new_name)
  new_name <- gsub("p12.raw_count.lapse", "lapse.p12.raw_count", new_name)
  new_name <- gsub("p24.raw_count.lapse", "lapse.p24.raw_count", new_name)
  new_name <- gsub("p48.raw_count.lapse", "lapse.p48.raw_count", new_name)
  new_name <- gsub("p72.raw_count.lapse", "lapse.p72.raw_count", new_name)
  new_name <- gsub("p168.raw_count.lapse", "lapse.p168.raw_count", new_name)
  new_name <- gsub("p12.diff_count.lapse", "lapse.p12.diff_count", new_name)
  new_name <- gsub("p24.diff_count.lapse", "lapse.p24.diff_count", new_name)
  new_name <- gsub("p48.diff_count.lapse", "lapse.p48.diff_count", new_name)
  new_name <- gsub("p72.diff_count.lapse", "lapse.p72.diff_count", new_name)
  new_name <- gsub("p168.diff_count.lapse", "lapse.p168.diff_count", new_name)
  new_name <- gsub("p12.raw_count.ema", "missing.p12.raw_count", new_name)
  new_name <- gsub("p24.raw_count.ema", "missing.p24.raw_count", new_name)
  new_name <- gsub("p48.raw_count.ema", "missing.p48.raw_count", new_name)
  new_name <- gsub("p72.raw_count.ema", "missing.p72.raw_count", new_name)
  new_name <- gsub("p168.raw_count.ema", "missing.p168.raw_count", new_name)
  new_name <- gsub("p12.diff_count.ema", "missing.p12.diff_count", new_name)
  new_name <- gsub("p24.diff_count.ema", "missing.p24.diff_count", new_name)
  new_name <- gsub("p48.diff_count.ema", "missing.p48.diff_count", new_name)
  new_name <- gsub("p72.diff_count.ema", "missing.p72.diff_count", new_name)
  new_name <- gsub("p168.diff_count.ema", "missing.p168.diff_count", new_name)
  return(new_name) 
}

feat_0 <- feat_0 |> 
  mutate(variable = fct_relabel(variable, clean_feature_names))

In [ ]:
nrow(shap_0)

[1] 77138381

[1] 77783328

Pull out one subject as example

In [ ]:
feat_test <- feat_0 |> 
  filter(id_obs == 1)

shap_test <- shap_0 |> 
  filter(id_obs == 1)

test <- feat_test |> 
  left_join(shap_test, by = c("id_obs", "variable"))

Not sure why there are two missing shap values for features for this observation?

For now combine will combine feature scores with shaps using right join

In [ ]:
shap_feat <- feat_0 |> 
  right_join(shap_0, by = c("id_obs", "variable")) |> 
  glimpse()

Rows: 77,138,381
Columns: 6
$ id_obs        <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,…
$ variable      <fct> lapse.p12.diff_count, lapse.p24.raw_count, lapse.p24.dif…
$ feature_score <dbl> 0.00000000, 0.00000000, 0.00000000, 0.00000000, 0.000000…
$ value         <dbl> -1.202097e-01, -7.540426e-03, -7.395879e-02, -3.579643e-…
$ rfvalue       <dbl> 0.016984509, -0.232523230, 0.025711884, -0.296854932, 0.…
$ mean_value    <dbl> 1.376266e-01, 9.903273e-03, 9.631549e-02, 5.314153e-02, …

### Standardize feature scores

In [ ]:
shap_feat <- shap_feat |> 
  group_by(variable) |> 
  mutate(feature_score_z = (feature_score - mean(feature_score))/sd(feature_score))

### Group categories

Sum of shap values and mean of z-scored feature values

In [ ]:
shap_feat_grp <- shap_feat |>  
    mutate(variable_grp = if_else(str_detect(variable, "lapse."),
                           "past use (EMA item)",
                           variable),
           variable_grp = if_else(str_detect(variable_grp, "ema_2"),
                           "craving (EMA item)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "ema_3"),
                           "past risky situation (EMA item)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "ema_4"),
                           "past stressful event (EMA item)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "ema_5"),
                           "past pleasant event (EMA item)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "ema_6"),
                           "valence (EMA item)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "ema_7"),
                           "arousal (EMA item)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "ema_8"),
                           "future risky situation (EMA item)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "ema_9"),
                           "future stressful event (EMA item)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "ema_10"),
                           "future efficacy (EMA item)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "missing."),
                           "missing surveys (other)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "day"),
                           "lapse day (other)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "hour"),
                           "lapse hour (other)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "age"),
                           "age (demographic)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "sex"),
                           "sex (demographic)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "marital"),
                           "marital (demographic)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "race"),
                           "race (demographic)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "educ"),
                           "education (demographic)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "income"),
                           "income (demographic)",
                           variable_grp),
           variable_grp = if_else(str_detect(variable_grp, "employ"),
                           "employment (demographic)",
                           variable_grp)) |> 
    mutate(variable_grp = factor(variable_grp)) |> 
    group_by(id_obs, variable_grp) |>  # values are already averaged across repeats
    summarize(value = sum(value),
              feature_score_z_mean = mean(feature_score_z))

`summarise()` has grouped output by 'id_obs'. You can override using the
`.groups` argument.

In [ ]:
 head(shap_feat_grp, n = 20)

# A tibble: 20 × 4
# Groups:   id_obs [1]
   id_obs variable_grp                           value feature_score_z_mean
    <dbl> <fct>                                  <dbl>                <dbl>
 1      1 age (demographic)                  0.000134                1.32  
 2      1 education (demographic)           -0.00122                -0.542 
 3      1 employment (demographic)           0.00507                 1.39  
 4      1 income (demographic)              -0.00604                -0.716 
 5      1 marital (demographic)              0.000137                0.204 
 6      1 race (demographic)                 0.0000241               0.386 
 7      1 sex (demographic)                 -0.00137                 0.978 
 8      1 future efficacy (EMA item)         0.350                   0.218 
 9      1 craving (EMA item)                -0.0143                  0.0230
10      1 past risky situation (EMA item)    0.0284                  0.889 
11      1 past stressful event (EMA item)   -0

### Save out tibble

In [ ]:
shap_feat_grp |> 
  write_rds(here::here(path_models_lag, "outer_shapsgrp_with_features_1week_0_v1_nested_main.rds")) 